## Chicago Food Inspections

### Data was downloaded from the [Chicago Data Portal](https://data.cityofchicago.org/Health-Human-Services/Food-Inspections-7-1-2018-Present/qizy-d2wf/about_data)

### 1. Import and Initial Clean-Up

Step includes adjusting formatting and removes any row for restaurants not in Illinois.
I've also added a 'month' and 'year' column to make querying easier.

In [1]:
import pandas as pd
import re
from rapidfuzz import fuzz
from collections import defaultdict
from tqdm import tqdm

df = pd.read_csv('Food_Inspections.csv')

df_clean = df.copy()

df_clean.columns = df_clean.columns.str.strip()
df_clean.columns = df_clean.columns.str.lower()


df_clean[['aka name', 'dba name','violations','city']] = df_clean[['aka name', 'dba name','violations','city']].replace(',', '', regex=True).replace("'","", regex=True).apply(lambda x: x.str.title())

df_clean['inspection date'] = pd.to_datetime(df_clean['inspection date'], format="%m/%d/%Y")
df_clean['zip'] = df_clean['zip'].fillna(0).astype('int64')
df_clean['license #'] = df_clean['license #'].fillna(0).astype('int64')
df_clean = df_clean[df_clean['license #'] !=0]
df_clean = df_clean[df_clean['state'] == 'IL']

df_clean['month'] = df_clean['inspection date'].dt.month
df_clean['year'] = df_clean['inspection date'].dt.year

df_clean.reset_index(drop=True, inplace=True)

### 2.  Updating and Cleaning 'Address' fields.

This code cleans the data in the 'address' column by:

1. Adjusting and standardizing format:
<div style="margin-left: 40px;">
    a.  'St.' vs 'st' vs 'ST' vs 'street' vs 'Street' vs 'STREET'
</div>
<div style="margin-left: 40px;">
    b.  '1234-1578' instead of '1234 - 1578'
</div>
<div style="margin-left: 40px;">
    c.  Directionals such as 'North' and 'n'
</div>

2. Groups locations that are likely the same by comparing text, with safeguards in place to ensure specific aspects must match exactly, such that addresses that are identical except for 'N' and 'S' are not grouped together.

3. Creates a column labeled 'cleaned address' to house this information rather than overwritting the original.

In [2]:
def clean_address(addr):
    if pd.isna(addr):
        return addr
    
    a = str(addr).lower().strip()
    a = a.replace('.', '')
    a = re.sub(r'\s*-\s*', '-', a)
    a = re.sub(r'\s+', ' ', a).strip()
    
    directionals = {
        r'\bnorth\b': 'n',
        r'\bsouth\b': 's',
        r'\beast\b': 'e',
        r'\bwest\b': 'w',
        r'\bnortheast\b': 'ne',
        r'\bnorthwest\b': 'nw',
        r'\bsoutheast\b': 'se',
        r'\bsouthwest\b': 'sw',
    }
    for pattern, replacement in directionals.items():
        a = re.sub(pattern, replacement, a)
    
    street_types = {
        r'\bstreet\b': 'st',
        r'\bavenue\b': 'ave',
        r'\bboulevard\b': 'blvd',
        r'\bdrive\b': 'dr',
        r'\bcourt\b': 'ct',
        r'\bcircle\b': 'cir',
        r'\blane\b': 'ln',
        r'\bplace\b': 'pl',
        r'\bplaza\b': 'plz',
        r'\bparkway\b': 'pkwy',
        r'\bhighway\b': 'hwy',
        r'\bsquare\b': 'sq',
        r'\bterrace\b': 'ter',
        r'\btrail\b': 'trl',
        r'\bway\b': 'way',
        r'\broad\b': 'rd',
        r'\bexpressway\b': 'expy',
        r'\bfreeway\b': 'fwy',
        r'\bturnpike\b': 'tpke',
        r'\bpike\b': 'pike',
        r'\balley\b': 'aly',
        r'\bpath\b': 'path',
        r'\bcrossing\b': 'xing',
        r'\bjunction\b': 'jct',
        r'\bstation\b': 'sta',
        r'\bmount\b': 'mt',
        r'\bmountain\b': 'mtn',
        r'\bpoint\b': 'pt',
        r'\bfort\b': 'ft',
        r'\bheights\b': 'hts',
        r'\bcenter\b': 'ctr',
        r'\bcorner\b': 'cor',
        r'\bmanor\b': 'mnr',
        r'\bvillage\b': 'vlg',
        r'\bcrescent\b': 'cres',
        r'\bcove\b': 'cv',
        r'\bmews\b': 'mews',
        r'\bbend\b': 'bnd',
        r'\brun\b': 'run',
        r'\bridge\b': 'rdg',
        r'\bpass\b': 'pass',
        r'\bloop\b': 'loop',
        r'\bcommon\b': 'cmn',
    }
    for pattern, replacement in street_types.items():
        a = re.sub(pattern, replacement, a)
    
    unit_types = {
        r'\bapartment\b': 'apt',
        r'\bsuite\b': 'ste',
        r'\bunit\b': 'unit',
        r'\bfloor\b': 'fl',
        r'\bbuilding\b': 'bldg',
        r'\bnumber\b': 'no',
        r'#\s*': '# '
    }
    for pattern, replacement in unit_types.items():
        a = re.sub(pattern, replacement, a)
        a = re.sub(r'\s+', ' ', a).strip()
    return a

df_clean["_address_clean"] = df_clean["address"].apply(clean_address)


ADDRESS_SIMILARITY_THRESHOLD = 90

address_groups_by_block = defaultdict(list)

DIRECTIONALS = {'n', 's', 'e', 'w', 'ne', 'nw', 'se', 'sw'}

def extract_numbers(addr):
    return re.findall(r'\d+', addr)

def extract_directionals(addr):
    return [token for token in addr.split() if token in DIRECTIONALS]

def find_or_create_group(addr):
    if pd.isna(addr):
        return addr

    block_key = get_block_key(addr)
    candidates = address_groups_by_block[block_key]
    addr_numbers = extract_numbers(addr)
    addr_directionals = extract_directionals(addr)

    best_score = 0
    best_group = None
    for group in candidates:
        if extract_numbers(group["canonical"]) != addr_numbers:
            continue
        if extract_directionals(group["canonical"]) != addr_directionals:
            continue
        score = fuzz.token_sort_ratio(addr, group["canonical"])
        if score > best_score:
            best_score = score
            best_group = group

    if best_score >= ADDRESS_SIMILARITY_THRESHOLD:
        best_group["members"].append(addr)
        return best_group["canonical"]
    else:
        new_group = {"canonical": addr, "members": [addr]}
        address_groups_by_block[block_key].append(new_group)
        return addr

tqdm.pandas(desc="Matching addresses")

def get_block_key(addr):
    match = re.match(r'^(\d+)', addr)
    return match.group(1) if match else addr[:3]

df_clean["cleaned address"] = df_clean["_address_clean"].progress_apply(find_or_create_group)

df_clean.drop(columns=["_address_clean"], inplace=True)

Matching addresses: 100%|██████████| 138013/138013 [00:01<00:00, 75053.96it/s]


### 3.  Addressing inconsistencies in License #'s and Names

This code cleans up the dataset where the same business may appear multiple times under slightly different names or license numbers, due to things like renewals, relicensing, or data entry inconsistencies. I used the newly created 'cleaned address' as a variable to ensure establishments with similar names did not get merged unless they also shared a common address.  The block of code:

1. Adjusts formatting to standardize the text.
2. Scans the data to find cases where the same name ('aka name') shows up with different license numbers, and consolidates them into a single consistent license number.
3. Picks 1 standard name to represent the business.
4. Performs a second scan within each address to catch near-identical names while keeping different businesses at the same address separate.
5. Creates 2 new columns for this updated information to be recorded ('cleaned license #' and 'cleaned name').

In [3]:
df_clean["_name_clean"]    = df_clean["aka name"].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
df_clean["_license_clean"] = df_clean["license #"].astype(str).str.strip().str.lower()

name_addr_to_license = {}
name_lic_to_license  = {}
addr_lic_to_license  = {}

for idx, row in df_clean.iterrows():
    n, a, l = row["_name_clean"], row["cleaned address"], row["_license_clean"]

    na_key = (n, a)
    nl_key = (n, l)
    al_key = (a, l)

    if na_key not in name_addr_to_license:
        name_addr_to_license[na_key] = row["license #"]
    if nl_key not in name_lic_to_license:
        name_lic_to_license[nl_key] = row["license #"]
    if al_key not in addr_lic_to_license:
        addr_lic_to_license[al_key] = row["license #"]

def resolve_license(row):
    n, a, l = row["_name_clean"], row["cleaned address"], row["_license_clean"]

    na_match = name_addr_to_license.get((n, a))
    nl_match = name_lic_to_license.get((n, l))
    al_match = addr_lic_to_license.get((a, l))

    if na_match is not None:
        return na_match
    elif nl_match is not None:
        return nl_match
    elif al_match is not None:
        return al_match
    else:
        return row["license #"]

df_clean["cleaned license #"] = df_clean.apply(resolve_license, axis=1)

license_to_first_name = {}
for idx, row in df_clean.iterrows():
    lic = row["cleaned license #"]
    if lic not in license_to_first_name:
        license_to_first_name[lic] = row["aka name"]

df_clean["cleaned name"] = df_clean["cleaned license #"].map(license_to_first_name)

SIMILARITY_THRESHOLD = 80

addr_to_entries = {}

for idx, row in df_clean.iterrows():
    a         = row["cleaned address"]
    cur_name  = row["_name_clean"]
    cur_lic   = row["cleaned license #"]
    cur_aka   = row["cleaned name"]

    if a not in addr_to_entries:
        addr_to_entries[a] = [(cur_name, cur_lic, cur_aka)]
    else:
        best_match = None
        best_score = 0
        for stored_name, stored_lic, stored_aka in addr_to_entries[a]:
            score = fuzz.token_sort_ratio(cur_name, stored_name)
            if score > best_score:
                best_score = score
                best_match = (stored_name, stored_lic, stored_aka)

        if best_score >= SIMILARITY_THRESHOLD:
            df_clean.at[idx, "cleaned license #"] = best_match[1]
            df_clean.at[idx, "cleaned name"]      = best_match[2]
        else:
            addr_to_entries[a].append((cur_name, cur_lic, cur_aka))

df_clean.drop(columns=["_name_clean", "_license_clean"], inplace=True)


### 3. Break Out and Clean 'Violations' column

The 'Violations' column is made up of any amount of violations and notes found during the inspection, making it impossible to determine what type of violations were found quickly and their individual risk level (note the pre-existing 'risk' column is not indicative of risk factors based on current inspection violations, instead it is to track for the potential risk of food-borne illnesses).

Using the [Foodborne Illness Risk Factors And Public Health Interventions](https://www.chicago.gov/city/en/depts/cdph/provdrs/food_safety/svcs/understand_healthcoderequirementsforfoodestablishments.html) breakdown established by Chicago, I was able to create a list of the categories and the violation(s) that exist under each one, and create a column that 

In [4]:
health_codes = [
    {"category":"Supervision", "breach":[1, 2]},
    {"category":"Employee Health", "breach":[3, 4, 5]},
    {"category":"Good Hygenic Practices", "breach":[6, 7]},
    {"category":"Preventing Contamination By Hands", "breach":[8, 9, 10]},
    {"category":"Approved Source", "breach":[11, 12, 13, 14]},
    {"category":"Protection From Contamination", "breach":[15, 16, 17]},
    {"category":"Time/Temperature Control For Safety", "breach":[18, 19, 20, 21, 22, 23, 24]},
    {"category":"Consumer Advisory", "breach" : [25]},
    {"category":"Highly Susceptible Populations", "breach": [26]},
    {"category":"Food/Color Additives and Toxic Substances", "breach": [27, 28]},
    {"category": "Comformance With Approved Procedures", "breach": [29]},
    {"category": "Safe Food And Water", "breach":[30, 31, 32]},
    {"category": "Food Temperature Control", "breach":[33, 34, 35, 36]},
    {"category": "Food Identification", "breach": [37]},
    {"category": "Prevention Of Food Contamination", "breach":[38, 39, 40, 41, 42]},
    {"category": "Proper Use Of Utensils", "breach":[43, 44, 45, 46]},
    {"category": "Utensils Equipment And Vending", "breach":[47, 48, 49]},
    {"category": "Physical Facilities", "breach":[50, 51, 52, 53, 54, 55, 56]},
    {"category": "Employee Training", "breach":[57, 58]},
    {"category": "City of Chicago Ordinance Compliance", "breach":[59, 60, 61, 62, 63]},
]

In [5]:
def extract_violation_nums(text):
    if pd.isna(text):
        return []
    segments = text.split('|')
    nums = []
    for seg in segments:
        match = re.match(r'\s*(\d{1,2})\.', seg)
        if match:
            nums.append(match.group(1))
    return nums

df_clean['violation nums'] = df_clean['violations'].apply(extract_violation_nums)

violation_to_category = {
    v: entry['category'] 
    for entry in health_codes 
    for v in entry['breach']
}

df_clean['violation categories'] = df_clean['violation nums'].apply(
    lambda nums: [violation_to_category.get(int(n), 'Unknown') for n in nums]
    if isinstance(nums, list) else []
)

violation_to_category = {
    v: entry['category'] 
    for entry in health_codes 
    for v in entry['breach']
}

df_clean['violation categories'] = df_clean['violation nums'].apply(
    lambda nums: [violation_to_category.get(int(n), 'Unknown') for n in nums]
    if isinstance(nums, list) else []
)


In [6]:
high_codes = {1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 21, 22, 27, 31, 32, 33, 50, 52}
med_codes = {10, 14, 15, 16, 17, 18, 19, 20, 23, 24, 25, 26, 28, 29, 30, 34, 35, 36, 37, 38, 39, 40, 41, 42, 47, 49, 51, 53}
low_codes = {43, 44, 45, 46, 48, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63}

priority_map = {}
for n in high_codes:
    priority_map[n] = 'high'
for n in med_codes:
    priority_map[n] = 'medium'
for n in low_codes:
    priority_map[n] = 'low'

df_clean['violation priorities'] = df_clean['violation nums'].apply(
    lambda nums: [priority_map.get(int(n), 'unknown') for n in nums]
    if isinstance(nums, list) else []
)

### 4. Clean up 'Facility Type' and create a 'Neighborhood' column

Facility Type included too many distinctions and typos or entry differences to be useful (Rooftop vs roof top vs Roof Tops for example).  I went ahead and created a list to replace entries with a cleaner version than some of what I found in the same column.

I created a 'neighborhood' column so the average person can easily identify the general location of each establishment.

**Notes:  
    1. the dataset includes zip codes for establishments outside of Illinois.  Any zipcode outside of the Chicago area does not have a 'neighborhood' entry.
    2. zip codes cover multiple neighborhoods in certain cases (such as 60605) - these zip codes will output all neighborhoods included in the zip.

In [7]:
df_clean['facility type'] = (
    df_clean['facility type']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)

df_clean['facility type'] = df_clean['facility type'].replace({
    "CHILDREN'S SERVICES FACILITY": "Children's Facility",
    "1023 CHILDREN'S SERVICES FACILITY": "Children's Facility",
    "1023-CHILDREN'S SERVICES FACILITY": "Children's Facility",
    "CHILDRENS SERVICES FACILITY": "Children's Facility",
    "AFTER SCHOOL PROGRAM": "Children's Facility",
    "BEFORE AND AFTER SCHOOL PROGRAM": "Children's Facility",
    "YOUTH HOUSING": "Children's Facility",
    "DAYCARE ABOVE AND UNDER 2 YEARS": "Daycare",
    "DAYCARE (2 - 6 YEARS)": "Daycare",
    "DAYCARE (UNDER 2 YEARS)": "Daycare",
    "DAYCARE COMBO 1586": "Daycare",
    "DAYCARE NIGHT": "Daycare",
    "DAYCARE 2 YRS TO 12 YRS": "Daycare",
    "DAY CARE 2-14": "Daycare",
    "15 MONTS TO 5 YEARS OLD": "Daycare",
    "DAYCARE": "Daycare",
    "CHARTER SCHOOL": "School",
    "PRIVATE SCHOOL": "School",
    "PUBLIC SHCOOL": "School",
    "CHARTER": "School",
    "CPS (CHARTER)": "School",
    "TEACHING SCHOOL": "School",
    "PASTRY SCHOOL": "School",
    "A-NOT-FOR-PROFIT CHEF TRAINING PROGRAM": "School",
    "HIGH SCHOOL KITCHEN": "School",
    "PREP INSIDE SCHOOL": "School",
    "SCHOOL CAFETERIA": "School",
    "UNIVERSITY CAFETERIA": "Cafeteria",
    "CHARTER SCHOOL CAFETERIA": "School",
    "COOKING SCHOOL": "Culinary School or Kitchen",
    "CULINARY SCHOOL": "Culinary School or Kitchen",
    "COOKING CLASS": "Culinary School or Kitchen",
    "CULINARY KITCHEN": "Culinary School or Kitchen",
    "CULINARY CLASS ROOMS": "Culinary School or Kitchen",
    "OUTREACH CULINARY KITCHEN": "Culinary School or Kitchen",
    "ADULT DAYCARE": "Supportive Living",
    "ASSISTED LIVING": "Supportive Living",
    "ASSISTED LIVING SENIOR CARE": "Supportive Living",
    "NURSING HOME": "Supportive Living",
    "SENIOR DAY CARE": "Supportive Living",
    "ADULT FAMILY CARE CENTER": "Supportive Living",
    "ADULT DAY SERVICE": "Supportive Living",
    "SUPPORTIVE LIVING": "Supportive Living",
    "MOBILE FOOD PREPARER": "Mobile or Pop-Up Establishment",
    "MOBILE FOOD DISPENSER": "Mobile or Pop-Up Establishment",
    "MOBILE PREPARED FOOD VENDOR": "Mobile or Pop-Up Establishment",
    "MOBILE FOOD": "Mobile or Pop-Up Establishment",
    "MOBILE PUSH CART": "Mobile or Pop-Up Establishment",
    "MOBILE DESSERTS VENDOR": "Mobile or Pop-Up Establishment",
    "MOBILE FOOD DESSERTS VENDOR": "Mobile or Pop-Up Establishment",
    "MFD TRUCK": "Mobile or Pop-Up Establishment",
    "MOBILPREPARED FOOD VENDOR": "Mobile or Pop-Up Establishment",
    "TRUCK": "Mobile or Pop-Up Establishment",
    "FROZEN DESSERT PUSHCARTS": "Mobile or Pop-Up Establishment",
    "POP-UP ESTABLISHMENT HOST-TIER I": "Mobile or Pop-Up Establishment",
    "POP-UP ESTABLISHMENT HOST-TIER II": "Mobile or Pop-Up Establishment",
    "POP-UP ESTABLISHMENT HOST-TIER III": "Mobile or Pop-Up Establishment",
    "POP-UP FOOD ESTABLISHMENT USER-TIER I": "Mobile or Pop-Up Establishment",
    "POP-UP FOOD ESTABLISHMENT USER-TIER II": "Mobile or Pop-Up Establishment",
    "SHARED KITCHEN USER (LONG TERM)": "Shared Kitchen",
    "SHARED KITCHEN USER (SHORT TERM)": "Shared Kitchen",
    "SHARED KITCHEN SUPPLEMENTAL": "Shared Kitchen",
    "TEST KITCHEN/ STORAGE": "Shared Kitchen",
    "KITCHEN DEMO": "Shared Kitchen",
    "CHARITY AID KITCHEN": "Shared Kitchen",
    "VENDING COMMISSARY": "Commissary",
    "COMMISSARY": "Commissary",
    "COMMIASARY": "Commissary",
    "COMMISARY": "Commissary",
    "COMISSARY": "Commissary",
    "TAVERN": "Tavern or Bar",
    "BAR": "Tavern or Bar",
    "TAVERN/BAR": "Tavern or Bar",
    "CATERED LIQUOR": "Tavern or Bar",
    "TAVERN/PACKAGED GOODS": "Tavern or Bar",
    "RESTAURANT/BAR": "Restaurant",
    "RESTAURANT/GROCERY STORE": "Restaurant",
    "RESTAURANT": "Restaurant",
    "DINING HALL": "Cafeteria",
    "CAFETERIA": "Cafeteria",
    "FOOD HALL": "Banquet",
    "FOOD BOOTH": "Special Event or Event Space",
    "BANQUET HALL": "Banquet",
    "BANQUET": "Banquet",
    "BANQUET HALL/CATERING": "Banquet",
    "BANQUET ROOM": "Banquet",
    "BANQUETS": "Banquet",
    "LOUNGE/BANQUET HALL": "Banquet",
    "CATERING/BANQUET": "Banquet",
    "CATERING/CAFE": "Banquet",
    "BANQUET/KITCHEN": "Banquet",
    "VFW HALL": "Banquet",
    "SPECIAL EVENT": "Special Event or Event Space",
    "EVENT CENTER": "Special Event or Event Space",
    "EVENT SPACE": "Special Event or Event Space",
    "EVENT VENUE": "Special Event or Event Space",
    "EVENT VENU": "Special Event or Event Space",
    "PRIVATE EVENT SPACE": "Special Event or Event Space",
    "ENTERTAINMENT VENUE": "Special Event or Event Space",
    "STADIUM": "Special Event or Event Space",
    "MUSIC VENUE": "Special Event or Event Space",
    "THEATER": "Special Event or Event Space",
    "THEATRE": "Special Event or Event Space",
    "MOVIE THEATER": "Special Event or Event Space",
    "MOVIE THEATRE": "Special Event or Event Space",
    "ROOFTOPS": "Rooftop",
    "ROOFTOP": "Rooftop",
    "ROOF TOPS": "Rooftop",
    "ROOF TOP": "Rooftop",
    "ROOF  TOP": "Rooftop",
    "WRIGLEY ROOFTOP": "Rooftop",
    "GROCERY STORE": "Grocery Store",
    "GROCERY STORE/GAS STATION": "Gas Station",
    "GAS STATION/GROCERY": "Grocery Store",
    "GROCERY & RESTAURANT": "Grocery Store",
    "GROCERY/RESTAURANT": "Grocery Store",
    "GROCERY STORE/RESTAURANT": "Grocery Store",
    "GROCERY/BAKERY": "Grocery Store",
    "GROCERY/TAQUERIA": "Grocery Store",
    "GROCERY/DELI": "Grocery Store",
    "GROCERY/DRUG STORE": "Grocery Store",
    "DRUG STORE/GROCERY": "Grocery Store",
    "GROCERY/LIQUOR STORE": "Grocery Store",
    "GROCERY/GAS STATION": "Grocery Store",
    "GROCERY(GAS STATION)": "Gas Station",
    "GROCERY STORE /PHARMACY": "Grocery Store",
    "DOLLAR STORE WITH GROCERY": "Grocery Store",
    "GROCERY/DOLLAR STORE": "Grocery Store",
    "SLAUGHTER HOUSE/ GROCERY": "Grocery Store",
    "GAS STATION": "Gas Station",
    "GAS STATION/CONVENIENCE STORE": "Gas Station",
    "GAS STATION/RESTAURANT": "Gas Station",
    "GAS STATION STORE": "Gas Station",
    "GAS STATION/STORE": "Gas Station",
    "GAS STATION FOOD STORE": "Gas Station",
    "GAS STATION/MINI MART": "Gas Station",
    "GAS/MINI MART": "Gas Station",
    "SERVICE GAS STATION": "Gas Station",
    "GAS STATION/FOOD": "Gas Station",
    "STORE": "Store",
    "RETAIL": "Store",
    "RETAIL FOOD": "Store",
    "RETAIL SALES": "Store",
    "RETAIL  SALES": "Store",
    "CONVENIENCE STORE": "Store",
    "CONVENIENT STORE": "Store",
    "CONVNIENCE STORE": "Store",
    "CONVENIENCE": "Store",
    "CONVENIENCE/GAS STATION": "Store",
    "CONVENIENCE/DRUG STORE": "Store",
    "GENERAL STORE": "Store",
    "DOLLAR STORE": "Store",
    "DOLLAR TREE": "Store",
    "TOBACCO STORE": "Store",
    "DRUG STORE": "Store",
    "PHARMACY": "Store",
    "DONUT SHOP": "Store",
    "HOT DOG STATION": "Store",
    "COFFEE SHOP": "Coffee Shop",
    "COFFEE ROASTER": "Coffee Shop",
    "COFFEE KIOSK": "Kiosk",
    "COFFEE AND/OR DRINKS": "Coffee Shop",
    "LIQUOR/COFFEE KIOSK": "Kiosk",
    "KIOSK": "Kiosk",
    "NAVY PIER KIOSK": "Kiosk",
    "NON-ALCOHOLIC DRINKS": "Non-Alcoholic Drinks",
    "MILK TEA": "Non-Alcoholic Drinks",
    "SHAKES/TEAS": "Non-Alcoholic Drinks",
    "HERBALIFE": "Wellness and Nutrional Food Establishment",
    "HERBAL LIFE": "Wellness and Nutrional Food Establishment",
    "HERBALIFE/ZUMBA": "Wellness and Nutrional Food Establishment",
    "HERBALIFE NUTRITION": "Wellness and Nutrional Food Establishment",
    "HERBAL LIFE SHOP": "Wellness and Nutrional Food Establishment",
    "HERBAL REMEDY": "Wellness and Nutrional Food Establishment",
    "HERBAL DRINKS": "Wellness and Nutrional Food Establishment",
    "HERBAL MEDICINE": "Wellness and Nutrional Food Establishment",
    "NUTRITION SHAKES": "Wellness and Nutrional Food Establishment",
    "NUTRITION..SMOOTHIES/SHAKES": "Wellness and Nutrional Food Establishment",
    "EXERCISE AND NUTRITION BAR": "Wellness and Nutrional Food Establishment",
    "HEALTH FOOD STORE": "Wellness and Nutrional Food Establishment",
    "HEALTH CARE STORE": "Wellness and Nutrional Food Establishment",
    "JUICE BAR": "Wellness and Nutrional Food Establishment",
    "JUICE BAR/GROCERY": "Wellness and Nutrional Food Establishment",
    "JUICE AND SALAD BAR": "Wellness and Nutritional Food Establishment",
    "SMOOTHIE BAR": "Wellness and Nutritional Food Establishment",
    "SPA": "Wellness and Nutrional Food Establishment",
    "HEALTH CLUB": "Wellness and Nutrional Food Establishment",
    "FITNESS CENTER": "Fitness or Gym",
    "GYM": "Fitness or Gym",
    "ICE CREAM SHOP": "Ice Cream",
    "ICE CREAM": "Ice Cream",
    "PALETERIA": "Ice Cream",
    "MOBILE FROZEN DESSERTS VENDOR": "Mobile or Pop-Up Establishment",
    "CANDY SHOP": "Candy",
    "CANDY STORE": "Candy",
    "CANDY/GELATO": "Candy",
    "BAKERY": "Bakery",
    "DELI/BAKERY": "Bakery",
    "WHOLESALE BAKERY": "Wholesale",
    "SUSHI COUNTER": "Sushi",
    "GROCERY(SUSHI PREP)": "Sushi",
    "LIQUOR STORE": "Liquor Store",
    "LIQUOR/GROCERY STORE": "Liquor Store",
    "LIQUOR/GROCERY": "Liquor Store",
    "LIQUOR/GROCERY STORE/BAR": "Liquor Store",
    "GROCERY & LIQUOR STORE": "Grocery Store",
    "1475 LIQUOR": "Liquor Store",
    "LIVE POULTRY": "Live Animals or Butcher",
    "LIVE POULTRY SLAUGHTER": "Live Animals or Butcher",
    "LIVE POULTRY SLAUGHTER FACILITY": "Live Animals or Butcher",
    "POULTRY SLAUGHTER": "Live Animals or Butcher",
    "CUSTOM POULTRY SLAUGHTER": "Live Animals or Butcher",
    "POULTRY SLAUGHTER FACILITY": "Live Animals or Butcher",
    "BUTCHER SHOP": "Live Animals or Butcher",
    "BUTCHER, DELI": "Live Animals or Butcher",
    "HOTEL": "Hotel",
    "ROOM SERVICE": "Hotel",
    "AIRPORT LOUNGE": "Airport",
    "HELICOPTER TERMINAL": "Airport",
    "REGULATED BUSINESS": "Business",
    "LIMITED BUSINESS": "Business",
    "WHOLESALE": "Wholesale",
    "WHOLESALE & RETAIL": "Wholesale",
    "BREWERY": "Brewery or Distillary",
    "DISTILLERY WITH TASTING ROOM": "Brewery or Distillary",
    "RESTAURANT(PROTEIN SHAKE BAR)": "Wellness and Nutrional Food Establishment",
    "PROTEIN SHAKE BAR": "Wellness and Nutrional Food Establishment",
    "CHURCH KITCHEN": "Church",
    "ARCHDIOCESE": "Church",
    "GYM STORE": "Wellness and Nutrional Food Establishment",
    "OTHER": "Other",
    "FLEA MARKET": "Other",
    "HAIR SALON": "Other",
    "NEWSSTAND": "Other",
    "ART CENTER": "Other",
    "RIVERWALK": "Other",
    "SHELTER": "Other",
    "REHAB CENTER": "Other",
    "NOT-FOR-PROFIT CLUB": "Other",
    "PRIVATE CLUB": "Other",
    "NIGHT CLUB": "Other",
    "HOOKA LOUNGE": "Other",
    "BLOCKBUSTER VIDEO": "Other",
    "DISTRIBUTION CENTER": "Other",
    "WAREHOUSE": "Other",
    "COLD STORAGE FACILITY": "Other",
    "GREENHOUSE": "Other",
    "REPACKAGING PLANT": "Other",
    "UNUSED STORAGE": "Other",
    "POPCORN CORN": "Other",
    "PREPACKAGE MEAL DISTRIBUTOR (1006 RETAIL)": "Other",
    "PACKAGED HEALTH FOODS": "Wellness and Nutrional Food Establishment",
    "SUSHI": "Sushi",
    "CHURCH": "Church",
    "AIRPORT": "Airport",
    "FITNESS OR GYM": "Fitness or Gym",
    "WELLNESS AND NUTRIONAL FOOD ESTABLISHMENT": "Wellness and Nutrional Food Establishment",
    "WELLNESS AND NUTRITIONAL FOOD ESTABLISHMENT":"Wellness and Nutrional Food Establishment",
    "SCHOOL": "School",
    "CHILDREN'S FACILITY": "Children's Facility",
    "MOBILE OR POP-UP ESTABLISHMENT":"Mobile or Pop-Up Establishment",
    "SPECIAL EVENT OR EVENT SPACE":"Special Event or Event Space",
    "LIVE ANIMALS OR BUTCHER":"Live Animals or Butcher",
    "BREWERY OR DISTILLARY":"Brewery or Distillary",
    "BUSINESS": "Business",
        })


### 5. Final clean and export (optional)

Additional cleanup to ensure a clean output should this file need to be used outside of Python - removed any commas, cleaned up the duplicate columns be replacing the old, and removing the original cleaned versions of that data.

In [8]:
cols = ['violation nums', 'violation categories','violation priorities']

df_clean[cols] = df_clean[cols].apply(
    lambda x: x.apply(lambda y: '; '.join(map(str, y)) if isinstance(y, list) else y))

df_clean[['aka name', 'license #', 'address']] = df_clean[['cleaned name', 'cleaned license #', 'cleaned address']].values
df_clean = df_clean.drop(columns=['cleaned name', 'cleaned license #', 'cleaned address'])

In [9]:
df_clean.to_csv('cleaned_chicago_food_inspections.csv', index=False, encoding='utf-8-sig')